<a href="https://colab.research.google.com/github/johanndeboda/AAI2026/blob/2026fall/ex2_react_codegen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2 — Code Generation with ReACT Prompting
**Tools:** Google Colab, Gemini API (google-genai SDK, gemini-2.5-flash, free tier)
**Method:** ReACT loop — Reason (plan) → Act (generate code) → Observe (run code + tests) → Fix (feed errors back), max 4 iterations
**Task:** summarize_orders(), a function that cleans messy order amounts and returns a summary

In [2]:
!pip install -q -U google-genai 2>/dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 24.9 MB/s eta 0:00:00


In [3]:
from google import genai
from google.colab import userdata
import json, time, re, textwrap, traceback

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-2.5-flash"

def ask(prompt):
    """Send a prompt to Gemini; retries on free-tier rate limits."""
    last_error = None
    for attempt in range(4):
        try:
            time.sleep(7)  # stay under the free-tier limit
            return client.interactions.create(model=MODEL, input=prompt).output_text.strip()
        except Exception as e:
            last_error = e
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                time.sleep(30)
                continue
            raise
    raise RuntimeError(f"Failed after 4 attempts. Last error: {last_error}")

def show(title, text):
    print(f"\n--- {title} ---")
    for line in str(text).splitlines():
        print(textwrap.fill(line, width=110, subsequent_indent="    ") if line.strip() else "")

In [4]:
VAGUE_TASK = "Write a Python function summarize_orders(orders) that totals the order amounts."

FULL_TASK = """Write a Python function `summarize_orders(orders)`.

Input: a list of dicts, each with keys "order_id" (str) and "amount".
"amount" may be a string like "$1,200.50" or " 45 ", a number like 30, None, "abc", "-20", or missing entirely.

Output: a dict with exactly these keys:
- "total": sum of valid amounts, rounded to 2 decimals (float)
- "valid_count": number of valid orders (int)
- "invalid_ids": order_ids with missing, non-numeric, or negative amounts, in input order (list)

Rules:
- Strip spaces, "$" and "," before converting.
- An empty list returns {"total": 0.0, "valid_count": 0, "invalid_ids": []}.
- Never crash on a bad row; a row without an "amount" key is invalid.
- Standard library only. No os, sys, subprocess, file, or network access.
- No print statements inside the function."""

In [5]:
def run_tests(fn):
    cases = [
        ([], {"total": 0.0, "valid_count": 0, "invalid_ids": []}),
        ([{"order_id": "A1", "amount": "$1,200.50"}, {"order_id": "A2", "amount": " 45 "}],
         {"total": 1245.5, "valid_count": 2, "invalid_ids": []}),
        ([{"order_id": "B1", "amount": None}, {"order_id": "B2", "amount": "abc"},
          {"order_id": "B3", "amount": "-20"}, {"order_id": "B4", "amount": 30}],
         {"total": 30.0, "valid_count": 1, "invalid_ids": ["B1", "B2", "B3"]}),
        ([{"order_id": "C1"}, {"order_id": "C2", "amount": "19.999"}],
         {"total": 20.0, "valid_count": 1, "invalid_ids": ["C1"]}),
    ]
    failures = []
    for i, (inp, expected) in enumerate(cases, 1):
        try:
            got = fn(inp)
            if got != expected:
                failures.append(f"Test {i} FAILED: input={inp} expected={expected} got={got}")
        except Exception as e:
            failures.append(f"Test {i} CRASHED: {type(e).__name__}: {e}")
    return failures

In [6]:
REACT_PROMPT = """You are a careful Python developer working in a ReACT loop:
REASON (plan) -> ACT (write code) -> OBSERVE (test results are sent back to you) -> FIX.

TASK:
{task}

PREVIOUS ATTEMPTS AND TEST RESULTS:
{history}

Respond in exactly this format:
<thought>
2-4 sentences: your plan and the edge cases you will handle. If there are previous attempts,
name the exact cause of the last failure and how this version fixes it.
</thought>
<code>
Only the complete function definition. No markdown fences, no example calls, no print statements.
</code>"""

In [7]:
BLOCKED = ["import os", "import sys", "subprocess", "open(", "eval(", "exec(", "__import__"]

def extract(tag, text):
    m = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.S)
    return m.group(1).strip() if m else ""

def react_loop(task, max_iters=4):
    history = ""
    for i in range(1, max_iters + 1):
        print(f"\n{'=' * 25} ITERATION {i} {'=' * 25}")

        # REASON + ACT
        reply = ask(REACT_PROMPT.format(task=task, history=history or "(none, first attempt)"))
        thought, code = extract("thought", reply), extract("code", reply)
        show("REASON", thought)
        show("ACT (generated code)", code)

        # OBSERVE: safety check -> run code -> run tests
        if not code:
            observation = "No <code> block found. Follow the response format exactly."
        elif any(b in code for b in BLOCKED):
            observation = "Code used a blocked operation. Standard library only, no system or file access."
        else:
            try:
                namespace = {}
                exec(code, namespace)
                fn = namespace.get("summarize_orders")
                failures = ["summarize_orders is not defined."] if fn is None else run_tests(fn)
                observation = "ALL TESTS PASSED" if not failures else "\n".join(failures)
            except Exception:
                observation = "Code failed to run:\n" + traceback.format_exc(limit=1)
        show("OBSERVE", observation)

        if observation == "ALL TESTS PASSED":
            print(f"\nSolved in {i} iteration(s).")
            return code

        # FIX: feed the code + results back into the next prompt
        history += f"\nAttempt {i} code:\n{code}\nResult:\n{observation}\n"

    print("\nNot solved within the iteration limit.")
    return None

## Run 1 (v1): vague prompt
One-line task, no rules. Expect test failures; the loop uses the observations to fix it.

In [8]:
v1_code = react_loop(VAGUE_TASK)


========================= ITERATION 1 =========================

--- REASON ---

--- ACT (generated code) ---
def summarize_orders(orders):
    total_amount = 0
    for order in orders:
        total_amount += order['amount']
    return total_amount

--- OBSERVE ---
Test 1 FAILED: input=[] expected={'total': 0.0, 'valid_count': 0, 'invalid_ids': []} got=0
Test 2 CRASHED: TypeError: unsupported operand type(s) for +=: 'int' and 'str'
Test 3 CRASHED: TypeError: unsupported operand type(s) for +=: 'int' and 'NoneType'
Test 4 CRASHED: KeyError: 'amount'

========================= ITERATION 2 =========================

--- REASON ---

--- ACT (generated code) ---
def summarize_orders(orders):
    result = {
        'total': 0.0,
        'valid_count': 0,
        'invalid_ids': []
    }

    for order in orders:
        order_id = order.get('id', 'unknown') # Safely get id for invalid_ids

        if 'amount' in order and isinstance(order['amount'], (int, float)):
            result['total'

## Run 2 (v2): full spec prompt
Same loop, with inputs, outputs, edge cases, allowed libraries, and error-handling rules spelled out.

In [9]:
v2_code = react_loop(FULL_TASK)


========================= ITERATION 1 =========================

--- REASON ---
I will initialize `total`, `valid_count`, and `invalid_ids`. For each order, I'll check for the "amount" key.
    If present, I'll process the value by stripping leading/trailing spaces, "$", and "," characters, then
    attempt to convert it to a float. If the conversion succeeds and the amount is non-negative, I'll add it
    to the total and increment the valid count; otherwise, the order's ID goes into `invalid_ids`. If the
    "amount" key is missing, the order's ID also goes into `invalid_ids`. The final total will be rounded to
    two decimal places.

--- ACT (generated code) ---
def summarize_orders(orders):
    total = 0.0
    valid_count = 0
    invalid_ids = []

    for order in orders:
        order_id = order.get("order_id") # Assuming order_id is always present if order dict exists

        amount_value = order.get("amount")

        is_valid_order = False
        if amount_value is not None

In [10]:
namespace = {}
exec(v2_code, namespace)
summarize_orders = namespace["summarize_orders"]

sample = [
    {"order_id": "X1", "amount": "$2,499.99"},
    {"order_id": "X2", "amount": "free"},
    {"order_id": "X3", "amount": 150},
    {"order_id": "X4"},
]
show("FINAL FUNCTION ON NEW DATA", summarize_orders(sample))


--- FINAL FUNCTION ON NEW DATA ---
{'total': 2649.99, 'valid_count': 2, 'invalid_ids': ['X2', 'X4']}


## Iteration notes

**v1 (vague prompt): not solved in 4 iterations**
- Iteration 1: summed raw values → crashed on strings, None, and the missing "amount" key (TypeError, KeyError).
- Iteration 2: used the test feedback to switch to .get() and a result dict, but guessed the wrong ID key ("id") → every invalid ID came back as "unknown".
- Iteration 3: fixed the ID key and added string cleaning, but counted -20 as valid and didn't round (19.999) because the prompt never stated those rules.
- Iteration 4: broke the response format (no code block).
- The model also skipped the REASON section every time.

**v2 (full spec prompt): solved in 1 iteration**
- Stating inputs, outputs, edge cases (negatives, missing key, rounding), allowed libraries, and the empty-list behavior let the model plan correctly up front, and its REASON step named every edge case before writing code.

**Takeaway:** the observe → fix loop works (each v1 fix came straight from the test errors), but it can't recover rules the prompt never gave. Specific constraints reduced 4+ iterations to 1.